In [1]:
import cv2
import numpy as np
import os

In [2]:
left_ref = cv2.imread('left/25.png')
right_ref = cv2.imread('right/244.png')
up_ref = cv2.imread('up/295.png')

In [3]:
def otsu_binarization(references):
    gray_references = []
    ret = []
    th = []
    
    for ref in references:
        gray = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY)
        r, t = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        gray_references.append(gray)
        ret.append(r)
        th.append(t)
    
    return ret, th


In [4]:
def component_resize(mask):
    kernel = np.ones((7,7), np.uint8)
    
    connectivity = 4 
    output = cv2.connectedComponentsWithStats(mask, connectivity, cv2.CV_32S)

    num_labels = output[0]
    labels = output[1]
    stats = output[2]
    
    max_area = 0
    max_idx = -1
    bbox = None
    
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        # print(area)
        if area > max_area:
            max_area = area
            max_idx = i
            bbox = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
            # print (max_area)
        
    left, top, width, height = bbox
    component = mask[top:top + height, left:left + width]
    resized_component = cv2.resize(component, (200, 200))
            
    # while True:
    #     cv2.imshow("mask", mask)

    #     key = cv2.waitKey(10) & 0xFF
    #     if (key == ord('q')):
    #         break
        # print (max_area)
    return resized_component

    

In [5]:
def calculate_iou(ref_mask, proc_mask):
    intersection = np.logical_and(ref_mask, proc_mask).sum()
    union = np.logical_or(ref_mask, proc_mask).sum()
    
    if union == 0:
        return 0
    
    return intersection / union

def classify_arrow(test_mask, reference_masks):
    iou_scores = [calculate_iou(test_mask, ref_mask) for ref_mask in reference_masks]
    print(f'iou = {max(iou_scores)}.2f, label = {np.argmax(iou_scores)}')
    return np.argmax(iou_scores)

In [6]:
def process_images(folder_path):
    images = []
    file_names = []
    for file_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file_name)
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            file_names.append(file_name)
    return images, file_names

In [7]:
def process_and_display_results(test_images, reference_masks, true_labels, file_names):
    labels_text = ["Left", "Right", "Up"]
    predictions = []
    
    for i, img in enumerate(test_images):
        _, binarized = otsu_binarization([img])
        resized_component = component_resize(binarized[0])
        
        if resized_component is not None:
            pred_label = classify_arrow(resized_component, reference_masks)
            predictions.append(pred_label)
            
            result_text = f"Predicted: {labels_text[pred_label]}"
            cv2.putText(img, result_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            
            print(f"Image {file_names[i]}: Predicted - {labels_text[pred_label]}, True - {labels_text[true_labels[i]]}")


    return predictions

In [8]:
def calculate_accuracy(predictions, true_labels, file_names):
    correct = 0
    incorrect_detections = []
    
    for i, (p, t) in enumerate(zip(predictions, true_labels)):
        if p == t:
            correct += 1
        else:
            incorrect_detections.append((file_names[i], p, t))
    
    accuracy = correct / len(true_labels) if true_labels else 0
    
    print("\nIncorrect detections:")
    for file_name, pred, true in incorrect_detections:
        print(f"File: {file_name}, Predicted: {pred}, True: {true}")
    
    return accuracy

In [9]:
reference_images = [left_ref, right_ref, up_ref]
_, reference_masks = otsu_binarization(reference_images)
reference_resized = [component_resize(mask) for mask in reference_masks]

left_images, left_names = process_images("left")
right_images, right_names = process_images("right")
up_images, up_names = process_images("up")

test_images = left_images + right_images + up_images
file_names = left_names + right_names + up_names
ground_truth = [0] * len(left_images) + [1] * len(right_images) + [2] * len(up_images)

predictions = process_and_display_results(test_images, reference_resized, ground_truth, file_names)

accuracy = calculate_accuracy(predictions, ground_truth, file_names)
print(f"Accuracy: {accuracy:.2f}")

iou = 0.9636457212370748.2f, label = 0
Image 0.png: Predicted - Left, True - Left
iou = 0.9589053977807034.2f, label = 0
Image 1.png: Predicted - Left, True - Left
iou = 0.9559079572725989.2f, label = 0
Image 10.png: Predicted - Left, True - Left
iou = 0.9252516699595447.2f, label = 0
Image 100.png: Predicted - Left, True - Left
iou = 0.9567534135781912.2f, label = 0
Image 101.png: Predicted - Left, True - Left
iou = 0.9401850627891606.2f, label = 0
Image 102.png: Predicted - Left, True - Left
iou = 0.9473733982694217.2f, label = 0
Image 103.png: Predicted - Left, True - Left
iou = 0.9504607200531966.2f, label = 0
Image 104.png: Predicted - Left, True - Left
iou = 0.9417913993822761.2f, label = 0
Image 105.png: Predicted - Left, True - Left
iou = 0.9392322583691393.2f, label = 0
Image 106.png: Predicted - Left, True - Left
iou = 0.9442541410539158.2f, label = 0
Image 107.png: Predicted - Left, True - Left
iou = 0.9469022356953392.2f, label = 0
Image 108.png: Predicted - Left, True - Le